---
# 06. FPI 지도 시각화

## 분석 목적

버거+샌드위치+패스트푸드 독립 브랜드의 FPI 분포를 지도에 시각화하여
업종 특화 관점에서 경쟁압력 공간 패턴을 확인한다.

restaurant FPI 지도와 비교하여 업종을 좁혔을 때
공간적 패턴이 어떻게 달라지는지 탐색한다.

| 단계 | 내용 |
|---|---|
| STEP 6-1 | 전체 독립 브랜드 FPI 지도 (연속형) |
| STEP 6-2 | FPI 구간별 지도 (NP/LP/HP) |
| STEP 6-3 | 생존/고전 브랜드 위치 지도 |

**입력 데이터**
- `biz_indie_with_groups_burger.csv`

**산출물**
- `fpi_map_burger.html`: FPI 수치 연속형
- `fpi_map_burger_group.html`: NP/LP/HP 구간별
- `fpi_map_burger_survival.html`: 생존/고전 브랜드 위치

## 공통 라이브러리 및 설정

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import warnings
warnings.filterwarnings('ignore')

PATH_to_data = "../results"
PATH_to_save = "../results"

# 데이터 로드
indie_groups = pd.read_csv(f"{PATH_to_data}/biz_indie_with_groups_burger.csv")
indie_groups = indie_groups[indie_groups['is_open'] == 1].copy()

indie_map = indie_groups.copy()
indie_map['fpi_group'] = indie_map['fpi_group'].fillna('NP')
group_label = {'NP': '무풍지대 (NP)', 'LP': '저압력 (LP)', 'HP': '고압력 (HP)'}
indie_map['구간'] = indie_map['fpi_group'].map(group_label)

color_map = {
    '무풍지대 (NP)': '#4a90e2',
    '저압력 (LP)':   '#f5a623',
    '고압력 (HP)':   '#e65100'
}

print(f"독립 브랜드: {len(indie_map):,}개")
print(f"\nFPI 구간 분포:")
print(indie_map['fpi_group'].value_counts())

독립 브랜드: 535개

FPI 구간 분포:
fpi_group
LP    219
HP    197
NP    119
Name: count, dtype: int64


---
## STEP 6-1. 전체 독립 브랜드 FPI 지도 (연속형)

In [7]:
fig = px.scatter_mapbox(
    indie_map,
    lat='latitude', lon='longitude',
    color='fpi_300m',
    color_continuous_scale='RdYlGn_r',
    range_color=[0, indie_map['fpi_300m'].quantile(0.90)],  # 상위 10% 기준으로 스케일
    size='review_count',
    size_max=20,
    hover_name='name',
    hover_data={
        'stars': True,
        'fpi_300m': ':.3f',
        '구간': True,
        'review_count': True,
        'neighborhood': True,
        'latitude': False,
        'longitude': False
    },
    zoom=11,
    height=700,
    title='Las Vegas 버거+샌드+패스트푸드 독립 브랜드 FPI 분포 (90th percentile 기준 스케일)'
)
fig.update_layout(
    mapbox_style='open-street-map',
    margin={'r': 0, 't': 50, 'l': 0, 'b': 0},
    coloraxis_colorbar=dict(
        title='FPI (경쟁압력)',
    )
)
fig.write_html(f"{PATH_to_save}/fpi_map_burger.html")
fig.show()
print(f"90th percentile FPI: {indie_map['fpi_300m'].quantile(0.90):.4f}")

90th percentile FPI: 4.6984


In [6]:
print("=== FPI 구간 분포 (영업 중 독립 브랜드) ===")
print(indie_map['fpi_group'].value_counts())
print(f"\n전체: {len(indie_map):,}개")
print(f"\n비율:")
print((indie_map['fpi_group'].value_counts() / len(indie_map) * 100).round(1))

print(f"\n=== FPI 수치 분포 ===")
print(indie_map['fpi_300m'].describe().round(4))
print(f"\nFPI = 0 (NP): {(indie_map['fpi_300m'] == 0).sum()}개")
print(f"FPI > 0: {(indie_map['fpi_300m'] > 0).sum()}개")

=== FPI 구간 분포 (영업 중 독립 브랜드) ===
fpi_group
LP    219
HP    197
NP    119
Name: count, dtype: int64

전체: 535개

비율:
fpi_group
LP    40.9
HP    36.8
NP    22.2
Name: count, dtype: float64

=== FPI 수치 분포 ===
count    535.0000
mean       1.8896
std        2.1308
min        0.0000
25%        0.3819
50%        1.2733
75%        2.7086
max       14.8886
Name: fpi_300m, dtype: float64

FPI = 0 (NP): 119개
FPI > 0: 416개


**지도가 고압력이 적어 보인 이유:**

지도 1에서 색상 스케일이 0~14.89까지 전체 범위로 설정돼 있어 대부분 업체가 FPI 0~3 사이에 몰려 있으니까 상대적으로 초록색(저압력)으로 보임
    
-> 색상 스케일 범위를 조정으로 해결

---
## STEP 6-2. FPI 구간별 지도 (NP/LP/HP)

In [3]:
fig2 = px.scatter_mapbox(
    indie_map,
    lat='latitude', lon='longitude',
    color='구간',
    color_discrete_map=color_map,
    size='review_count',
    size_max=20,
    hover_name='name',
    hover_data={
        'stars': True,
        'fpi_300m': ':.3f',
        '구간': True,
        'review_count': True,
        'neighborhood': True,
        'latitude': False,
        'longitude': False
    },
    zoom=11,
    height=700,
    title='Las Vegas 버거+샌드+패스트푸드 독립 브랜드 FPI 구간 지도'
)
fig2.update_layout(
    mapbox_style='open-street-map',
    margin={'r': 0, 't': 50, 'l': 0, 'b': 0}
)
fig2.write_html(f"{PATH_to_save}/fpi_map_burger_group.html")
print("저장 완료: fpi_map_burger_group.html")
fig2.show()

저장 완료: fpi_map_burger_group.html


---
## STEP 6-3. 생존/고전 브랜드 위치 지도

HP 구간 내 생존/고전 브랜드의 공간적 분포를 시각화한다.
생존 브랜드와 고전 브랜드가 같은 상권 내에서도
어떻게 다르게 분포하는지 확인한다.

In [4]:
# 생존/고전 브랜드 정의 (is_open 전체 포함)
MIN_REVIEW = 10
hp_all = indie_groups[indie_groups['fpi_group'] == 'HP'].copy()

survivor_map = hp_all[
    (hp_all['stars'] >= 4.0) &
    (hp_all['review_count'] >= MIN_REVIEW)
].copy()
struggle_map = hp_all[
    (hp_all['stars'] <= 3.0) &
    (hp_all['review_count'] >= MIN_REVIEW)
].copy()

survivor_map['그룹'] = '생존 (4.0↑)'
struggle_map['그룹']  = '고전 (3.0↓)'
hp_map = pd.concat([survivor_map, struggle_map], ignore_index=True)

print(f"생존 브랜드: {len(survivor_map)}개")
print(f"고전 브랜드: {len(struggle_map)}개")

survival_color = {
    '생존 (4.0↑)': '#2ecc71',
    '고전 (3.0↓)': '#e74c3c'
}

fig3 = px.scatter_mapbox(
    hp_map,
    lat='latitude', lon='longitude',
    color='그룹',
    color_discrete_map=survival_color,
    size='review_count',
    size_max=20,
    hover_name='name',
    hover_data={
        'stars': True,
        'fpi_300m': ':.3f',
        '그룹': True,
        'review_count': True,
        'neighborhood': True,
        'latitude': False,
        'longitude': False
    },
    zoom=11,
    height=700,
    title='Las Vegas HP 구간 생존/고전 브랜드 위치 지도\n(버거+샌드+패스트푸드)'
)
fig3.update_layout(
    mapbox_style='open-street-map',
    margin={'r': 0, 't': 50, 'l': 0, 'b': 0}
)
fig3.write_html(f"{PATH_to_save}/fpi_map_burger_survival.html")
print("저장 완료: fpi_map_burger_survival.html")
fig3.show()

생존 브랜드: 63개
고전 브랜드: 65개
저장 완료: fpi_map_burger_survival.html


In [8]:
print("=== 지도 3 필터링 단계별 업체 수 ===")
print(f"전체 독립 브랜드 (영업 중): {len(indie_map):,}개")
print(f"HP 구간만: {(indie_map['fpi_group']=='HP').sum():,}개")

hp_all = indie_map[indie_map['fpi_group'] == 'HP'].copy()
print(f"HP + 리뷰수 10개↑: {(hp_all['review_count']>=10).sum():,}개")
print(f"  → 생존 (별점 4.0↑): {((hp_all['stars']>=4.0) & (hp_all['review_count']>=10)).sum():,}개")
print(f"  → 고전 (별점 3.0↓): {((hp_all['stars']<=3.0) & (hp_all['review_count']>=10)).sum():,}개")
print(f"  → 중간 (3.0~4.0):   {((hp_all['stars']>3.0) & (hp_all['stars']<4.0) & (hp_all['review_count']>=10)).sum():,}개")

=== 지도 3 필터링 단계별 업체 수 ===
전체 독립 브랜드 (영업 중): 535개
HP 구간만: 197개
HP + 리뷰수 10개↑: 180개
  → 생존 (별점 4.0↑): 63개
  → 고전 (별점 3.0↓): 65개
  → 중간 (3.0~4.0):   52개


---
## STEP 6 결과 정리

### 공간적 패턴 요약

| 패턴 | 내용 |
|---|---|
| Strip/Paradise 집중 | HP 구간이 Strip 중심부에 압도적으로 집중 |
| 외곽 분산 | NP/LP가 주거지역(Spring Valley, Westside)으로 분산 |
| 생존/고전 혼재 | 동일 HP 상권 안에서 생존/고전 브랜드가 혼재 |

### restaurant FPI 지도와의 차이

| 항목 | 전체 Restaurants | 버거+샌드+패스트푸드 |
|---|---|---|
| 전체 업체 수 | 4,818개 | 535개 |
| HP 집중도 | Strip + 교통축 선형 패턴 | Strip 중심 집중 (선형 패턴 약화) |
| NP 분포 | 외곽 전체 분산 | 외곽 소수 존재 |
| 해석 | 다양한 업종으로 넓게 분포 | 버거 업종 특성상 Strip 의존도 높음 |

> 버거+샌드+패스트푸드 업종은 restaurant 대비 Strip 의존도가 더 높다.
> 이는 해당 업종이 관광객 유동인구에 더 민감하게 반응하는
> 업종 특성을 반영한다.

### 생존/고전 브랜드 공간 패턴

생존 브랜드와 고전 브랜드가 동일한 HP 상권 내에 혼재한다.
이는 입지 자체보다 **브랜드 전략(시그니처 메뉴, 수제·오너십)**이
생존을 결정하는 핵심 요인임을 공간적으로 확인해준다.